In [ ]:
# Install missing dependencies if needed
%pip install statsmodels ucimlrepo

# Model Comparison with Inferential Statistics

This notebook compares the performance of multiple machine learning models on the heart disease dataset using both predictive metrics and inferential statistical plotting approaches. You will:

- Train several models (Logistic Regression, Random Forest, SVM, etc.)
- Evaluate and compare their performance
- Use statistical plots and tests to interpret differences


## 1. Import Required Libraries

We import libraries for data manipulation, machine learning, statistical analysis, and plotting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)


## 2. Load and Explore the Dataset

Load the heart disease dataset and perform basic EDA.

In [ ]:
# Load Heart Disease dataset from Hugging Face (already downloaded to data folder)

from datasets import load_from_disk

ds = load_from_disk('../data/heart_disease_hf')

# If the dataset has splits, use 'train' or merge all

if 'train' in ds:
    df = ds['train'].to_pandas()
else:
    # If only one split, convert directly
    df = ds.to_pandas()

# Inspect columns and target

print('Columns:', df.columns.tolist())

# Assume target column is named 'target' or similar

if 'target' not in df.columns:
    # Try common alternatives
    for col in ['Target', 'target', 'HeartDisease', 'heart_disease', 'output']:
        if col in df.columns:
            df['target'] = df[col]
            break
    else:
        raise ValueError('Target column not found. Please check dataset columns.')

# Basic EDA

print('Shape:', df.shape)

display(df.head())

display(df.describe())

sns.countplot(x='target', data=df)

plt.title('Class Distribution')

plt.show()

## 3. Preprocess the Data

Handle missing values, encode categoricals, and scale features.

In [ ]:
# Impute missing values (median for numeric)
for col in df.select_dtypes('number').columns:
    df[col].fillna(df[col].median(), inplace=True)

# Encode categoricals if any (none in Cleveland, but template for general use)
for col in df.select_dtypes('object').columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Split features/target
X = df.drop('target', axis=1).values
y = df['target'].values

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)


In [ ]:
# Remove any rows with NaNs after imputation (robustness for all columns)
mask = ~np.isnan(X).any(axis=1)
X = X[mask]
y = y[mask]
print('After NaN removal:', X.shape, y.shape)

## 4. Split Data into Train and Test Sets

Split the data for model training and evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 5. Train Multiple Models

Train Logistic Regression, Random Forest, and SVM classifiers.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f'{name} trained.')

## 6. Evaluate Model Performance

Evaluate each model using accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix.

In [ ]:
results = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'ROC-AUC': roc
    })
    print(f'\n{name}\n', classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
results_df = pd.DataFrame(results)

## 7. Compare Model Metrics with Statistical Plots

Visualize the performance metrics of all models using boxplots, violin plots, and bar charts.

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
for metric in metrics:
    plt.figure(figsize=(7, 4))
    sns.barplot(x='Model', y=metric, data=results_df)
    plt.title(f'Model Comparison: {metric}')
    plt.ylim(0, 1)
    plt.show()

## 8. Perform Inferential Statistical Tests on Model Results

Apply paired t-tests and ANOVA to compare model performance.

In [ ]:
# Example: Paired t-test for ROC-AUC between models
from itertools import combinations
print('Paired t-tests for ROC-AUC:')
for (m1, m2) in combinations(results_df['Model'], 2):
    auc1 = results_df.loc[results_df['Model'] == m1, 'ROC-AUC'].values
    auc2 = results_df.loc[results_df['Model'] == m2, 'ROC-AUC'].values
    t_stat, p_val = stats.ttest_rel(auc1, auc2)
    print(f'{m1} vs {m2}: t={t_stat:.3f}, p={p_val:.3g}')

# ANOVA for F1 scores
f1s = [results_df.loc[results_df['Model'] == m, 'F1'].values for m in results_df['Model']]
f_stat, p_val = stats.f_oneway(*f1s)
print(f'ANOVA for F1: F={f_stat:.3f}, p={p_val:.3g}')

## 9. Visualize Statistical Comparisons

Plot confidence intervals and p-values to interpret the statistical significance of model comparisons.

In [ ]:
# Plot confidence intervals for F1 scores
import matplotlib.patches as mpatches
plt.figure(figsize=(7, 4))
for i, row in results_df.iterrows():
    mean = row['F1']
    ci = 1.96 * np.std([mean]) / np.sqrt(1)  # Placeholder for CI (single run)
    plt.bar(row['Model'], mean, yerr=ci, capsize=10)
plt.ylabel('F1 Score')
plt.title('Model F1 Scores with 95% CI (single run)')
plt.show()

# Note: For true CIs, use repeated cross-validation and aggregate results.
